# 스마트 창고 출고 지연 예측 v9

### v8 → v9 원인 분석 및 전략

| 버전 | OOF MAE | Public Score |
|---|---|---|
| v6 | 8.7129 | **베스트** |
| v7 | 8.7184 | 하락 |
| v8 | ~8.71 | v6보다 약간 하락 |

**v8 악화 원인 3가지**:
1. 신규 피처(slot_progress_x_*, sc_double_bottleneck 등) → train 과적합
2. 12개 모델 SLSQP → 8개보다 최적화 어려움
3. XGB RMSE→MAE 전환 → 다양성(diversity) 감소

**v9 전략: v6 완전 동일 + CB Optuna만 50→100 trials로 강화**
- v6 피처: 완전 동일 (변경 없음)
- v6 모델: 완전 동일 (M1~M8, Huber 포함)
- v6 XGB: RMSE 유지 (다양성 보존)
- CB Optuna: 30 → **50 trials** (CB가 0.414 가중치로 가장 중요!)
- 앙상블: SLSQP 동일

## 0. 임포트 + 설정

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import warnings

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED    = 42
N_FOLDS = 5
TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
POWER   = 0.4
np.random.seed(SEED)

# v6 LGB 최적 파라미터 하드코딩 (50 trials 결과)
BEST_LGB = dict(
    n_estimators      = 2478,
    learning_rate     = 0.016478723775160582,
    max_depth         = 7,
    num_leaves        = 61,
    subsample         = 0.7444928430592912,
    colsample_bytree  = 0.5056775143285802,
    reg_alpha         = 0.24040098526779663,
    reg_lambda        = 6.427489016616686,
    min_child_samples = 78,
)
print('v9 환경 설정 완료')
print(f'LGB: n_est={BEST_LGB["n_estimators"]}, lr={BEST_LGB["learning_rate"]:.4f}')

v9 환경 설정 완료
LGB: n_est=2478, lr=0.0165


## 1. 데이터 로드 + 레이아웃 (v6 동일)

In [2]:
train  = pd.read_csv('./data/train.csv')
test   = pd.read_csv('./data/test.csv')
layout = pd.read_csv('./data/layout_info.csv')

layout_type_map = {v: i for i, v in enumerate(layout['layout_type'].unique())}
layout['layout_type_enc'] = layout['layout_type'].map(layout_type_map)

layout_feats = [c for c in layout.columns if c not in ['layout_id','layout_type','layout_type_enc']]
scaler = StandardScaler()
layout_scaled = scaler.fit_transform(layout[layout_feats].fillna(0))
km = KMeans(n_clusters=8, random_state=SEED, n_init=10)
layout['layout_cluster'] = km.fit_predict(layout_scaled)

train = train.merge(layout, on='layout_id', how='left')
test  = test.merge(layout,  on='layout_id', how='left')

for mask, df in [(train['layout_cluster'].isna(), train),
                 (test['layout_cluster'].isna(),  test)]:
    if mask.any():
        for feat in layout_feats:
            if feat in df.columns:
                df.loc[mask, feat] = df.loc[mask, feat].fillna(df[feat].median())
        df.loc[mask, 'layout_cluster']  = 8
        df.loc[mask, 'layout_type_enc'] = -1

print(f'train: {train.shape}, test: {test.shape}')
print(f'unseen layouts - train: {train["layout_cluster"].isna().sum()}, test: {test["layout_cluster"].isna().sum()}')

train: (250000, 110), test: (50000, 109)
unseen layouts - train: 0, test: 0


## 2. 피처 엔지니어링 (v6 완전 동일)

In [3]:
def engineer_features_v6(df):
    d = df.copy()
    d['charge_pressure']       = d['charge_queue_length'] / (d['charger_count'] + 1)
    d['active_robot_ratio']    = d['robot_active'] / (d['robot_total'] + 1)
    d['idle_robot_ratio']      = d['robot_idle'] / (d['robot_total'] + 1)
    d['low_batt_robot_count']  = d['low_battery_ratio'] * d['robot_active']
    d['battery_stress']        = (1 - d['battery_mean'] / 100) * d['battery_std']
    d['effective_robot_avail'] = d['robot_total'] - d['low_batt_robot_count'] - d['charge_queue_length']
    d['robot_fault_ratio']     = d['fault_count_15m'] / (d['robot_active'] + 1)
    d['incident_total_15m']    = d['blocked_path_15m'] + d['near_collision_15m'] + d['fault_count_15m']
    d['congestion_load']       = d['congestion_score'] * d['avg_trip_distance']
    d['wait_per_intersection'] = d['intersection_wait_time_avg'] / (d['intersection_count'] + 1)
    d['path_congestion_gap']   = d['path_optimization_score'] - d['congestion_score']
    d['aisle_density']         = d['aisle_traffic_score'] / (d['aisle_width_avg'] + 0.1)
    d['order_complexity']      = d['unique_sku_15m'] * d['avg_items_per_order']
    d['urgent_heavy_ratio']    = d['urgent_order_ratio'] * d['heavy_item_ratio']
    d['effective_order_load']  = d['order_inflow_15m'] * (1 + d['urgent_order_ratio'])
    d['rework_pressure']       = d['return_order_ratio'] + d['replenishment_overlap']
    d['pick_complexity']       = d['pick_list_length_avg'] * (1 - d['sku_concentration'])
    d['dock_pack_util_avg']      = (d['pack_utilization'] + d['loading_dock_util'] + d['staging_area_util']) / 3
    d['orders_per_pack_station'] = d['order_inflow_15m'] / (d['pack_station_count'] + 1)
    d['orders_per_robot']        = d['order_inflow_15m'] / (d['robot_active'] + 1)
    d['robot_density']           = d['robot_total'] / (d['floor_area_sqm'] + 1)
    d['pack_area_pressure']      = d['pack_utilization'] * d['layout_compactness']
    d['dock_truck_bottleneck']   = d['loading_dock_util'] * d['outbound_truck_wait_min']
    d['it_bottleneck']         = d['wms_response_time_ms'] * (1 + d['scanner_error_rate'])
    d['barcode_fail_rate']     = 1 - d['barcode_read_success_rate']
    d['label_scan_bottleneck'] = d['label_print_queue'] * (1 + d['scanner_error_rate'])
    d['conveyor_load']         = d['avg_package_weight_kg'] / (d['conveyor_speed_mps'] + 0.01)
    d['orders_per_staff']      = d['order_inflow_15m'] / (d['staff_on_floor'] + 1)
    d['shift_load_ratio']      = d['order_inflow_15m'] / (d['prev_shift_volume'] + 1)
    d['handover_pressure']     = d['shift_handover_delay_min'] * d['prev_shift_volume'] / 1000
    d['shift_hour_sin'] = np.sin(2 * np.pi * d['shift_hour'] / 24)
    d['shift_hour_cos'] = np.cos(2 * np.pi * d['shift_hour'] / 24)
    d['dow_sin']        = np.sin(2 * np.pi * d['day_of_week'] / 7)
    d['dow_cos']        = np.cos(2 * np.pi * d['day_of_week'] / 7)
    d['is_peak_hour']   = d['shift_hour'].isin([9,10,11,14,15,16,17]).astype(int)
    d['area_per_robot']          = d['floor_area_sqm'] / (d['robot_total'] + 1)
    d['crossdock_dock_pressure'] = d['cross_dock_ratio'] * d['loading_dock_util']
    d['robot_per_pack_station']  = d['robot_total'] / (d['pack_station_count'] + 1)
    d['robot_order_congestion']  = d['orders_per_robot'] * d['congestion_score']
    d['battery_order_pressure']  = d['battery_stress'] * d['effective_order_load']
    d['low_batt_x_congestion']   = d['low_battery_ratio'] * d['congestion_score']
    d['charge_q_x_orders']       = d['charge_queue_length'] * d['order_inflow_15m']
    d['pack_saturated']         = (d['pack_utilization'] > 0.95).astype(int)
    d['pack_overflow_pressure'] = np.maximum(d['pack_utilization'] - 0.95, 0) * d['order_inflow_15m']
    d['pack_util_sq']           = d['pack_utilization'] ** 2
    d['order_robot_balance']   = d['order_inflow_15m'] / (d['robot_active'] * d['robot_utilization'] + 1)
    d['charge_demand_ratio']   = d['charge_queue_length'] / (d['robot_charging'] + 1)
    d['pack_order_ratio']      = d['pack_utilization'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['congestion_per_robot']  = d['congestion_score'] / (d['robot_active'] + 1)
    d['fault_per_100_orders']  = d['fault_count_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['block_per_100_orders']  = d['blocked_path_15m'] / (d['order_inflow_15m'] / 100 + 0.01)
    d['dock_order_match']      = d['loading_dock_util'] / (d['order_inflow_15m'] / 200 + 0.01)
    d['staff_order_match']     = d['staff_on_floor'] / (d['order_inflow_15m'] / 50 + 0.01)
    d['battery_adequacy']      = d['battery_mean'] / (100 - d['low_battery_ratio'] * 100 + 1)
    d['trip_efficiency']       = d['avg_trip_distance'] / (d['robot_utilization'] + 0.01)
    return d

train_fe = engineer_features_v6(train)
test_fe  = engineer_features_v6(test)
print(f'피처 엔지니어링 완료: {train_fe.shape}')

피처 엔지니어링 완료: (250000, 165)


## 3. 래그 + 롤링 피처 (v6 동일)

In [4]:
LAG_COLS  = ['order_inflow_15m','congestion_score','robot_utilization','battery_mean',
              'charge_queue_length','blocked_path_15m','fault_count_15m',
              'pack_utilization','loading_dock_util','effective_order_load','orders_per_robot']
CUM_COLS  = ['order_inflow_15m','congestion_score','blocked_path_15m','fault_count_15m','incident_total_15m']
ROLL_COLS = ['order_inflow_15m','congestion_score','battery_mean','pack_utilization',
             'robot_utilization','loading_dock_util']

def add_temporal(df, lag_cols, cum_cols, roll_cols):
    d = df.copy().sort_values(['scenario_id','shift_hour']).reset_index(drop=True)
    d['slot_idx']      = d.groupby('scenario_id').cumcount()
    d['slot_progress'] = d['slot_idx'] / 24
    for col in lag_cols:
        if col not in d.columns: continue
        l1 = d.groupby('scenario_id')[col].shift(1)
        l2 = d.groupby('scenario_id')[col].shift(2)
        d[f'{col}_lag1']       = l1; d[f'{col}_lag2'] = l2
        d[f'{col}_diff1']      = d[col] - l1
        d[f'{col}_pct_change'] = (d[col] - l1) / (l1.abs() + 1)
    for col in cum_cols:
        if col not in d.columns: continue
        d[f'{col}_cumsum'] = d.groupby('scenario_id')[col].cumsum()
    for col in roll_cols:
        if col not in d.columns: continue
        d[f'{col}_roll3_mean'] = d.groupby('scenario_id')[col].transform(
            lambda x: x.rolling(3, min_periods=1).mean())
        d[f'{col}_roll3_std']  = d.groupby('scenario_id')[col].transform(
            lambda x: x.rolling(3, min_periods=1).std())
    for col in ['order_inflow_15m','congestion_score','pack_utilization','battery_mean']:
        if col not in d.columns: continue
        d[f'{col}_exp_mean'] = d.groupby('scenario_id')[col].transform(
            lambda x: x.expanding().mean())
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        if col not in d.columns: continue
        def rolling_slope(x):
            out = np.full(len(x), np.nan); vals = x.values
            for i in range(2, len(vals)):
                yv = vals[i-2:i+1]
                if np.all(np.isfinite(yv)): out[i] = (yv[2]-yv[0])/2
            return pd.Series(out, index=x.index)
        d[f'{col}_trend3'] = d.groupby('scenario_id')[col].transform(rolling_slope)
    if 'pack_utilization_lag1' in d.columns:
        d['pack_sat_lag1'] = (d['pack_utilization_lag1'] > 0.95).astype(int)
    if 'order_inflow_15m_cumsum' in d.columns:
        d['cum_order_per_robot'] = d['order_inflow_15m_cumsum'] / (d['robot_total'] + 1)
    for col in ['order_inflow_15m','congestion_score','pack_utilization']:
        rm = f'{col}_roll3_mean'
        if rm in d.columns:
            d[f'{col}_deviation']       = d[col] - d[rm]
            d[f'{col}_deviation_ratio'] = d[f'{col}_deviation'] / (d[rm].abs() + 1)
    fill_cols = [c for c in d.columns if any(t in c for t in
                 ['_lag','_diff','_cumsum','_roll3','_exp_','_trend','_sat_lag','_pct_change','_deviation'])]
    for col in fill_cols:
        d[col] = d.groupby('scenario_id')[col].transform(lambda x: x.fillna(x.median()))
    d[fill_cols] = d[fill_cols].fillna(0)
    return d

train_fe = add_temporal(train_fe, LAG_COLS, CUM_COLS, ROLL_COLS)
test_fe  = add_temporal(test_fe,  LAG_COLS, CUM_COLS, ROLL_COLS)
print(f'전체 피처: {len([c for c in train_fe.columns if c not in ID_COLS+[TARGET]])}')

전체 피처: 239


## 4. 시나리오 통계 + 타겟 인코딩 (v6 동일)

In [5]:
SC_COLS = ['order_inflow_15m','congestion_score','robot_utilization',
            'battery_mean','pack_utilization','loading_dock_util']
for col in SC_COLS:
    for df in [train_fe, test_fe]:
        df[f'sc_{col}_mean'] = df.groupby('scenario_id')[col].transform('mean')
        df[f'sc_{col}_dev']  = df[col] - df[f'sc_{col}_mean']

def target_encode_cv(tr, te, col, target, n_splits=5, alpha=50):
    gm = tr[target].mean(); enc = np.zeros(len(tr))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for ti, vi in kf.split(tr):
        s = tr.iloc[ti].groupby(col)[target].agg(['mean','count'])
        s['sm'] = (s['count']*s['mean']+alpha*gm)/(s['count']+alpha)
        enc[vi] = tr.iloc[vi][col].map(s['sm']).fillna(gm)
    fs = tr.groupby(col)[target].agg(['mean','count'])
    fs['sm'] = (fs['count']*fs['mean']+alpha*gm)/(fs['count']+alpha)
    return enc, te[col].map(fs['sm']).fillna(gm).values

for col_name, feat_name in [('layout_cluster','te_layout_cluster'),('layout_type_enc','te_layout_type')]:
    tr_e, te_e = target_encode_cv(train_fe, test_fe, col_name, TARGET)
    train_fe[feat_name] = tr_e; test_fe[feat_name] = te_e
print('시나리오 통계(12개) + 타겟 인코딩 완료')

시나리오 통계(12개) + 타겟 인코딩 완료


## 5. 피처 준비

In [6]:
feature_cols = [c for c in train_fe.columns
                if c not in ID_COLS+[TARGET] and pd.api.types.is_numeric_dtype(train_fe[c])]
for col in feature_cols:
    med = train_fe[col].median()
    train_fe[col] = train_fe[col].fillna(med)
    test_fe[col]  = test_fe[col].fillna(med)
drop_cols = [c for c in feature_cols if train_fe[c].nunique()<=1 or train_fe[c].std()<1e-10]
feature_cols = [c for c in feature_cols if c not in drop_cols]
print(f'최종 피처: {len(feature_cols)}개')

X = train_fe[feature_cols].copy(); y = train_fe[TARGET].copy(); X_test = test_fe[feature_cols].copy()
y_tr_sqrt  = np.sqrt(y.clip(lower=0)); y_tr_log = np.log1p(y.clip(lower=0))
y_tr_power = np.power(y.clip(lower=0)+1e-8, POWER)
def decode_sqrt(p):  return (p.clip(0))**2
def decode_log(p):   return np.expm1(p).clip(0)
def decode_power(p): return np.power(p.clip(0), 1/POWER)
gkf = GroupKFold(n_splits=N_FOLDS); groups = train_fe['scenario_id']
print(f'왜도: 원본={stats.skew(y):.2f}, sqrt={stats.skew(y_tr_sqrt):.2f}, log={stats.skew(y_tr_log):.2f}')

최종 피처: 252개
왜도: 원본=5.68, sqrt=1.47, log=0.08


## 6. CB Optuna — **50 trials** (v6: 30 → v9: 50)

**v9 핵심 변경점**: CB가 SLSQP에서 0.414 가중치로 가장 중요 → 더 최적화
- 더 많은 trials로 더 좋은 파라미터 탐색
- Huber delta도 탐색 범위에 추가

In [ ]:
tidx = np.random.choice(len(X), 50000, replace=False)
Xt   = X.iloc[tidx].reset_index(drop=True)
yt_s = y_tr_sqrt.iloc[tidx].reset_index(drop=True)
yt_r = y.iloc[tidx].reset_index(drop=True)
gt   = groups.iloc[tidx].reset_index(drop=True)

def cb_obj(trial):
    p = dict(
        iterations       = trial.suggest_int('iterations', 500, 3000),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        depth            = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg      = trial.suggest_float('l2_leaf_reg', 0.5, 100, log=True),
        #subsample        = trial.suggest_float('subsample', 0.4, 0.95),
        min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 20, 600),
        random_strength  = trial.suggest_float('random_strength', 0.1, 10, log=True),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 2.0),
    )
    F = dict(loss_function='MAE', eval_metric='MAE', bootstrap_type='Bayesian',
             random_seed=SEED, early_stopping_rounds=30, verbose=0,
             task_type='GPU', devices='0')
    ms = []
    for ti, vi in GroupKFold(n_splits=3).split(Xt, yt_s, groups=gt):
        m = CatBoostRegressor(**p, **F)
        m.fit(Xt.iloc[ti], yt_s.iloc[ti], eval_set=(Xt.iloc[vi], yt_s.iloc[vi]))
        ms.append(mean_absolute_error(yt_r.iloc[vi], decode_sqrt(m.predict(Xt.iloc[vi]))))
    return np.mean(ms)

print('CB Optuna (50 trials + 확장 탐색 범위)...')
study_cb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study_cb.optimize(cb_obj, n_trials=50, show_progress_bar=True)
BEST_CB = study_cb.best_params
print(f'CB 최적 MAE: {study_cb.best_value:.4f}  (v6: 8.8924)')
print(f'파라미터: {BEST_CB}')

CB Optuna (50 trials + 확장 탐색 범위)...


  0%|          | 0/50 [00:00<?, ?it/s]

Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric perio

## 7. M1: LGB (sqrt) — v6 동일

In [ ]:
oof_lgb_sqrt, test_lgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
lgb_imp = np.zeros(len(feature_cols))
print('M1: LGB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(200)])
    oof_lgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    lgb_imp         += m.feature_importances_ / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_sqrt[vi]):.4f}')
print(f'M1 OOF MAE: {mean_absolute_error(y, oof_lgb_sqrt):.4f}')

## 8. M2: CB (sqrt)

In [ ]:
oof_cb_sqrt, test_cb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M2: CB(sqrt) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bayesian', random_seed=SEED,
                          early_stopping_rounds=50, verbose=200,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=(X.iloc[vi], y_tr_sqrt.iloc[vi]))
    oof_cb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_cb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_cb_sqrt[vi]):.4f}')
print(f'M2 OOF MAE: {mean_absolute_error(y, oof_cb_sqrt):.4f}')

## 9. M3: XGB (sqrt, RMSE) — v6 동일 유지
다양성을 위해 RMSE 그대로 유지

In [ ]:
oof_xgb_sqrt, test_xgb_sqrt = np.zeros(len(X)), np.zeros(len(X_test))
print('M3: XGB(sqrt, RMSE) 5-Fold  — v6 동일')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    print(f'-- Fold {fold+1} --')
    m = XGBRegressor(
        n_estimators=2000, learning_rate=0.03, max_depth=6,
        subsample=0.7, colsample_bytree=0.5,
        reg_alpha=5, reg_lambda=5, min_child_weight=100,
        device='cuda', tree_method='hist',
        random_state=SEED, verbosity=0
    )
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti],
          eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])], verbose=200)
    oof_xgb_sqrt[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_xgb_sqrt   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'   MAE: {mean_absolute_error(y.iloc[vi], oof_xgb_sqrt[vi]):.4f}')
print(f'M3 OOF MAE: {mean_absolute_error(y, oof_xgb_sqrt):.4f}')

## 10. M4: LGB (log1p)

In [ ]:
oof_lgb_log, test_lgb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M4: LGB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_log.iloc[ti], eval_set=[(X.iloc[vi], y_tr_log.iloc[vi])],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_lgb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_log[vi]):.4f}')
print(f'M4 OOF MAE: {mean_absolute_error(y, oof_lgb_log):.4f}')

## 11. M5: CB (log1p)

In [ ]:
oof_cb_log, test_cb_log = np.zeros(len(X)), np.zeros(len(X_test))
print('M5: CB(log) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = CatBoostRegressor(**BEST_CB, loss_function='MAE', eval_metric='MAE',
                          bootstrap_type='Bayesian', random_seed=SEED,
                          early_stopping_rounds=50, verbose=0,
                          task_type='GPU', devices='0')
    m.fit(X.iloc[ti], y_tr_log.iloc[ti], eval_set=(X.iloc[vi], y_tr_log.iloc[vi]))
    oof_cb_log[vi] = decode_log(m.predict(X.iloc[vi]))
    test_cb_log   += decode_log(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_cb_log[vi]):.4f}')
print(f'M5 OOF MAE: {mean_absolute_error(y, oof_cb_log):.4f}')

## 12. M6: LGB (power=0.4)

In [ ]:
oof_lgb_pow, test_lgb_pow = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M6: LGB(power={POWER}) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_power.iloc[ti], eval_set=[(X.iloc[vi], y_tr_power.iloc[vi])],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_pow[vi] = decode_power(m.predict(X.iloc[vi]))
    test_lgb_pow   += decode_power(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_pow[vi]):.4f}')
print(f'M6 OOF MAE: {mean_absolute_error(y, oof_lgb_pow):.4f}')

## 13. M7: LGB (Huber, delta=1.5) ← v6 베스트 개별 모델

In [ ]:
oof_lgb_huber, test_lgb_huber = np.zeros(len(X)), np.zeros(len(X_test))
print('M7: LGB(Huber delta=1.5) 5-Fold')
for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
    m = LGBMRegressor(**BEST_LGB, objective='huber', huber_delta=1.5,
                      device='gpu', random_state=SEED, verbose=-1)
    m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
          callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_lgb_huber[vi] = decode_sqrt(m.predict(X.iloc[vi]))
    test_lgb_huber   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    print(f'  Fold {fold+1} MAE: {mean_absolute_error(y.iloc[vi], oof_lgb_huber[vi]):.4f}')
print(f'M7 OOF MAE: {mean_absolute_error(y, oof_lgb_huber):.4f}  (v6: 8.7451)')

## 14. M8: LGB Multi-seed (3 seeds) — v6 동일

In [ ]:
SEEDS = [42, 123, 456]
oof_lgb_ms, test_lgb_ms = np.zeros(len(X)), np.zeros(len(X_test))
print(f'M8: LGB Multi-seed ({len(SEEDS)} seeds)')
for seed in SEEDS:
    oof_tmp, test_tmp = np.zeros(len(X)), np.zeros(len(X_test))
    for fold, (ti, vi) in enumerate(gkf.split(X, y, groups=groups)):
        m = LGBMRegressor(**BEST_LGB, objective='mae', device='gpu',
                          random_state=seed, verbose=-1,
                          subsample_seed=seed, feature_fraction_seed=seed)
        m.fit(X.iloc[ti], y_tr_sqrt.iloc[ti], eval_set=[(X.iloc[vi], y_tr_sqrt.iloc[vi])],
              callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
        oof_tmp[vi] = decode_sqrt(m.predict(X.iloc[vi]))
        test_tmp   += decode_sqrt(m.predict(X_test)) / N_FOLDS
    oof_lgb_ms  += oof_tmp  / len(SEEDS)
    test_lgb_ms += test_tmp / len(SEEDS)
    print(f'  Seed {seed}: OOF MAE = {mean_absolute_error(y, oof_tmp):.4f}')
print(f'M8 OOF MAE: {mean_absolute_error(y, oof_lgb_ms):.4f}')

## 15. 앙상블 — SLSQP (8개 모델, v6 동일 구성)

In [ ]:
oofs  = [oof_lgb_sqrt, oof_cb_sqrt, oof_xgb_sqrt, oof_lgb_log,
          oof_cb_log, oof_lgb_pow, oof_lgb_huber, oof_lgb_ms]
tests = [test_lgb_sqrt, test_cb_sqrt, test_xgb_sqrt, test_lgb_log,
          test_cb_log, test_lgb_pow, test_lgb_huber, test_lgb_ms]
names = ['LGB(sqrt)','CB(sqrt)','XGB(sqrt)','LGB(log)',
         'CB(log)','LGB(pow)','LGB(Huber)','LGB(ms)']
n_m = len(oofs)

print('=== 개별 모델 OOF MAE ===')
for name, oof in zip(names, oofs):
    print(f'  {name:>14}: {mean_absolute_error(y, oof):.4f}')

def ens_loss(w): return mean_absolute_error(y, sum(w[i]*oofs[i] for i in range(n_m)))
res = minimize(ens_loss, [1/n_m]*n_m, method='SLSQP',
               bounds=[(0,1)]*n_m,
               constraints={'type':'eq','fun':lambda w:sum(w)-1})
weights_slsqp = res.x

print(f'\nSLSQP OOF MAE: {res.fun:.4f}  (v6: 8.7129)')
print(f'v6 대비 개선: {8.7129 - res.fun:+.4f}')
print('\n가중치:')
for n, w in sorted(zip(names, weights_slsqp), key=lambda x:-x[1]):
    if w > 0.005: print(f'  {n:>14}: w={w:.3f}')

## 16. 최종 예측 + 제출

In [ ]:
final_preds = sum(weights_slsqp[i]*tests[i] for i in range(n_m)).clip(0)

print('=== 최종 예측 분포 ===')
print(f'  mean={final_preds.mean():.2f}, std={final_preds.std():.2f}')
print(f'  95th={np.percentile(final_preds,95):.1f}, 99th={np.percentile(final_preds,99):.1f}, max={final_preds.max():.1f}')

sub = pd.read_csv('./data/sample_submission.csv').drop(columns=[TARGET], errors='ignore')
sub = sub.merge(pd.DataFrame({'ID':test_fe['ID'], TARGET:final_preds}), on='ID', how='left')
sub.to_csv('./submission_v9.csv', index=False)
print('\nsubmission_v9.csv 저장 완료')
print(sub[TARGET].describe())

try:
    v6 = pd.read_csv('./submission_v6.csv')
    print(f'\nv6 mean={v6[TARGET].mean():.2f} → v9 mean={final_preds.mean():.2f}')
    print(f'v6 max={v6[TARGET].max():.1f} → v9 max={final_preds.max():.1f}')
except: pass

## 17. Feature Importance

In [ ]:
imp_df = pd.DataFrame({'feature':feature_cols,'importance':lgb_imp}).sort_values('importance',ascending=False).reset_index(drop=True)
print('=== Top 20 Features ===')
for _, row in imp_df.head(20).iterrows():
    print(f"  {row['feature']:45s} {row['importance']:8.1f}")